# 02. Classical ML Baseline Models for Code Vulnerability Detection

## 📌 Notebook Overview
In this notebook, we build and evaluate classical Machine Learning baseline models:
- **Feature Extraction**: Tokenizing source code and computing TF-IDF n-gram vectors (character & word n-grams).
- **Models**:
  1. **Logistic Regression** (Linear baseline)
  2. **XGBoost Classifier** (Gradient boosted tree baseline)
- **Evaluation**: Precision, Recall, F1-Score, and Confusion Matrix across vulnerability categories.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv("../data/sample_vulnerability_dataset.csv")

# Encode multi-class CWE labels
label_encoder = LabelEncoder()
df['cwe_label'] = label_encoder.fit_transform(df['cwe_category'])
class_names = label_encoder.classes_

print("Classes mapping:")
for idx, name in enumerate(class_names):
    print(f"  {idx}: {name}")

In [1]:
# Stratified Train / Validation / Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    df['function_code'], 
    df['cwe_label'], 
    test_size=0.25, 
    random_state=42, 
    stratify=df['cwe_label']
)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

In [1]:
# TF-IDF Feature Extraction (Combination of Word and Character n-grams for Code Syntax)
vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3),
    token_pattern=r'(?u)\w+|[^\w\s]',
    max_features=500
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

In [1]:
# Baseline Model 1: Logistic Regression
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

y_pred_lr = lr_model.predict(X_test_tfidf)

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_p, lr_r, lr_f1, _ = precision_recall_fscore_support(y_test, y_pred_lr, average='weighted')

print(f"--- Logistic Regression Performance ---")
print(f"Accuracy : {lr_acc:.4f}")
print(f"Precision: {lr_p:.4f}")
print(f"Recall   : {lr_r:.4f}")
print(f"F1-Score : {lr_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=class_names))

In [1]:
# Baseline Model 2: XGBoost Classifier
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    eval_metric='mlogloss'
)
xgb_model.fit(X_train_tfidf, y_train)

y_pred_xgb = xgb_model.predict(X_test_tfidf)

xgb_acc = accuracy_score(y_test, y_pred_xgb)
xgb_p, xgb_r, xgb_f1, _ = precision_recall_fscore_support(y_test, y_pred_xgb, average='weighted')

print(f"--- XGBoost Classifier Performance ---")
print(f"Accuracy : {xgb_acc:.4f}")
print(f"Precision: {xgb_p:.4f}")
print(f"Recall   : {xgb_r:.4f}")
print(f"F1-Score : {xgb_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=class_names))

In [1]:
# Plotting Confusion Matrix for Baseline Models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title("Logistic Regression Confusion Matrix", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title("XGBoost Classifier Confusion Matrix", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()

## 💡 Summary of Baseline Results
- **Logistic Regression**: Serves as a solid linear baseline, capturing explicit keyword patterns like `strcpy`, `SELECT * FROM`, `sprintf`, `malloc`.
- **XGBoost Classifier**: Captures non-linear feature combinations and code structural n-grams effectively.
- Next, we proceed to Transformer Fine-tuning (`03_transformer_finetune.ipynb`) to evaluate how contextual code embeddings compare against TF-IDF n-gram baselines.